# Random Fourier features for the wristband loss

What this measures, and what it does not.

The wristband loss pushes a batch of latents toward `N(0, I)` by spreading them out in the
`(direction, radius)` coordinates. The spreading term is the expensive one: written out, it
compares every point with every other, which costs `O(N^2 d)`.

Two ways around that are in the library. The **spectral** path cuts the spherical-harmonic series
at degree `l <= 1` and costs `O(N d K)`. The **Fourier** path samples the angular factor with
random features instead of cutting it, and costs `O(N D d)`.

The difference between them is the *kind* of error. A cut is a **bias**: several genuinely
different arrangements of the directions get the same value and the same gradient, so more
training steps never separate them. Sampling is **unbiased at every degree**, so the error is
variance, which falls as `1/D` and averages out over steps.

Four questions, in order:

1. Does the estimator agree with the identity the Lean development proves, and is it unbiased?
2. Where does it become cheaper than the exact kernel, in wall-clock time?
3. How many features does a wanted accuracy need — proved, analytic and measured, side by side?
4. In training, how close to the target does each path get, and how fast in seconds?

**On the judge.** Closeness to the target is scored with statistics no path trains against: an
energy distance for the direction, a KS distance for the radius, and a distance correlation
between the two. Scoring with the loss's own kernel would hand one path its objective as the exam.
The first two moments are not enough on their own: a batch can have mean zero and covariance `I`
and still sit on a small set.

## 1. Setup

`PRESET = "smoke"` runs in a few minutes on a laptop. `PRESET = "full"` uses a batch above the
crossover and wants a GPU. Results are cached to `results/`, so re-running a cell after its
measurement redraws instantly.

**Where the code comes from.** The next cell looks for `ml-tidbits/python` on disk first and
clones only if it finds nothing. The clone route needs the branch to exist on the remote; while
`fourier-features` is still a local branch, set `ML_TIDBITS` to the path of your checkout and the
clone is skipped.

In [ ]:
# --- settings ---------------------------------------------------------------------------------
PRESET          = "smoke"   # "smoke" (a few minutes) or "full" (large batch, wants a GPU)
FORCE_RECOMPUTE = False     # True re-runs every measurement and overwrites the cache

# Where the ml-tidbits code comes from. Setting ML_TIDBITS to a path overrides everything else,
# which is how you would point at a Google Drive mount. Otherwise the notebook looks on disk and,
# failing that, clones. The URL is the fork: the original repo carries no fourier path.
ML_TIDBITS        = None
ML_TIDBITS_REPO   = "https://github.com/andremiguelc/ml-tidbits"
ML_TIDBITS_BRANCH = "fourier-features"

# Only the tests' own keyword arguments appear here. Everything that defines the experiment --
# the model, the data generator, lr, the loss weights, k_modes, embed_dim, in_dim, the judge --
# stays at the value the repo uses.
SIZES = {
   "smoke": dict(
      batch_size   = 4096,                        # the N the repulsion sees, per step
      n_features   = 384,                         # D
      n_samples    = 40_960,
      n_epochs     = 4,
      seeds        = (0, 1),
      cal_reps     = 64,                          # see the note below on where these go
      time_sizes   = (512, 2048, 8192),
      var_features = (32, 128, 384, 1024, 4096),  # the D sweep behind the accuracy table
      var_reps     = 64,
      check_n      = 512,                          # the batch the two correctness checks use
      check_reps   = 100,
      in_dim       = 64,                           # must stay above embed_dim: the latent is a
      embed_dim    = 8,                            # bottleneck, or reconstruction applies no pressure
      hidden       = 64,
   ),
   "full": dict(
      batch_size   = 8192,                        # well above the crossover: this one is the point
      n_features   = 1024,
      n_samples    = 163_840,                     # 16 steps an epoch, so training still converges
      n_epochs     = 12,
      seeds        = (0, 1, 2),                   # each seed costs four whole trainings
      cal_reps     = 32,
      time_sizes   = (2048, 8192, 32768),
      var_features = (32, 128, 384, 1024, 4096),
      var_reps     = 128,
      check_n      = 512,
      check_reps   = 400,
      in_dim       = 256,                          # a wide latent needs wide data to compress
      embed_dim    = 64,
      hidden       = 256,
   ),
   # Where the time goes. TestFourierTrainingRun runs two modes: `shared calibration` over every
   # seed, and `own calibration` over the first seed only. That is 4 paths * (len(seeds) + 1)
   # whole trainings. Separately it rebuilds a loss inside the innermost loop, and each build
   # spends `cal_reps` forward passes measuring calibration constants; in the shared mode the
   # next line overwrites them, so that work is discarded. Hence a low `cal_reps` and few seeds,
   # and hence `batch_size` and `n_samples` left alone -- they are what the result is about.
}
CFG = SIZES[PRESET]

PATHS  = ("pairwise", "spectral", "fourier paired", "fourier phase")
TIME_D = 128                # the latent width the kernel timing and variance cells use

# "mlp" keeps the loss a real share of the step, which every speed number depends on. "full" is
# the attention-and-flow model that ships; run it to confirm the latent geometry, and ignore the
# speed columns while you do, because that encoder takes most of the step. The printed loss share
# says which case you are in.
MODEL = "mlp"

# The pairwise path holds several N-by-N float32 tensors through the backward pass. Above this
# budget it is skipped, and the skip is printed rather than left silent -- a missing column reads
# as a win if nobody says why it is missing.
MAX_PAIRWISE_GIB = 2.0

print(f"preset {PRESET}")
for key, value in CFG.items():
   print(f"   {key:<13} {value}")

In [ ]:
import importlib.machinery, importlib.util
import json, math, subprocess, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt


def _FindOnDisk(start: Path):
   """`ml-tidbits/python` at or above `start`, or None. The local route: no network, no clone."""
   for p in [start, *start.parents]:
      if (p / "ml-tidbits" / "python").is_dir():
         return p / "ml-tidbits" / "python"
   return None


def _Clone(into: Path) -> Path:
   """Clone the branch carrying the fourier path. The Colab route. An existing clone is reused."""
   target = into / "ml-tidbits"
   if not (target / "python").is_dir():
      subprocess.run(["git", "clone", "--depth", "1", "--branch", ML_TIDBITS_BRANCH,
                      ML_TIDBITS_REPO, str(target)], check=True)
   return target / "python"


_CWD = Path.cwd().resolve()
_ON_DISK = _FindOnDisk(_CWD)
if ML_TIDBITS:
   ML_PATH, _ROUTE = Path(ML_TIDBITS).expanduser(), "set by hand"
elif _ON_DISK is not None:
   ML_PATH, _ROUTE = _ON_DISK, "found on disk"
else:
   ML_PATH, _ROUTE = _Clone(_CWD), f"cloned {ML_TIDBITS_REPO} at {ML_TIDBITS_BRANCH}"
if str(ML_PATH) not in sys.path:
   sys.path.insert(0, str(ML_PATH))


def _BindTestsPackage(tests_dir: Path) -> None:
   """Point the name `tests` at this checkout, whatever else on the machine answers to it.

   Neither `tests` nor `embed_models` holds an __init__.py, so both are namespace packages. A
   regular package of the same name anywhere on sys.path beats a namespace portion no matter the
   order, and `tests` is a common enough name that a hosted runtime supplies one. Nothing is
   called `embed_models`, which is why only this half needs help. The helpers import each other
   as `tests.X`, so a plain by-path load would not be enough.
   """
   for name in [m for m in list(sys.modules) if m == "tests" or m.startswith("tests.")]:
      del sys.modules[name]
   spec = importlib.machinery.ModuleSpec("tests", None, is_package=True)
   spec.submodule_search_locations = [str(tests_dir)]
   sys.modules["tests"] = importlib.util.module_from_spec(spec)


_BindTestsPackage(ML_PATH / "tests")

from embed_models.EmbedModels import C_WristbandGaussianLoss
from tests.TestFourierTraining import TestFourierTrainingRun
from tests.TestFourierWristband import (TestFourierFactoredIdentity, TestFourierUnbiased,
                                        TestFourierSeesWhatSpectralCannot, TestFourierTiming,
                                        TestFourierVarianceAndD)

# Fail here with an instruction, rather than deep inside a figure. The original ml-tidbits repo
# has no fourier path at all, so aiming this at the wrong repo or branch is an easy mistake.
try:
   C_WristbandGaussianLoss(repulsion="fourier", reduction="global", calibration_shape=None)
except (TypeError, ValueError) as _err:
   raise RuntimeError(
      f"this checkout of ml-tidbits has no fourier path ({_err}). Point ML_TIDBITS_BRANCH at the "
      "branch that carries it, or ML_TIDBITS at a checkout that does.") from _err

NB_DIR  = Path.cwd()
FIG_DIR = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
RES_DIR = NB_DIR / "results"; RES_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- style ------------------------------------------------------------------------------------
# One colour per path, the same in every figure. Grey is the exact path, so it stays neutral.
COLOR = {"pairwise": "#8a8a8a", "spectral": "#e08214",
         "fourier paired": "#2a5d9f", "fourier phase": "#7fb3e0"}

mpl.rcParams.update({
   "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
   "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
   "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.6,
   "axes.spines.top": False, "axes.spines.right": False,
   "axes.axisbelow": True,
   "legend.frameon": False, "legend.fontsize": 9,
   "lines.linewidth": 2.0, "lines.markersize": 6,
   "figure.facecolor": "white", "axes.facecolor": "white",
})


# --- cache ------------------------------------------------------------------------------------

def _Jsonable(o):
   if hasattr(o, "tolist"):
      return o.tolist()
   if isinstance(o, (np.floating, np.integer)):
      return o.item()
   raise TypeError(f"cannot store {type(o)} in the cache")


def cached(name, produce):
   """Run `produce` once and keep the answer, keyed on the preset.

   The stored value is read back even on a fresh run, so a cache hit and a fresh run give the
   same object and a figure cannot behave differently on the second pass. Delete one file in
   `results/` to recompute one measurement.
   """
   path = RES_DIR / f"{PRESET}__{name}.json"
   if path.exists() and not FORCE_RECOMPUTE:
      print(f"[cache] {path.name}")
      return json.loads(path.read_text())
   out = produce()
   path.write_text(json.dumps(out, indent=1, default=_Jsonable))
   return json.loads(path.read_text())


def _PairGiB(batch):
   """Roughly what the pairwise kernel alone costs at this batch. Memory goes as the square."""
   return 8. * 4. * float(batch) ** 2 / 2. ** 30


print(f"ml-tidbits: {_ROUTE}\n   {ML_PATH}")
print(f"device: {DEVICE}   torch {torch.__version__}")
print(f"pairwise needs about {_PairGiB(CFG['batch_size']):.1f} GiB at batch "
      f"{CFG['batch_size']}, before the model's own activations")

## 2. Does the estimator do what the proof says

Two checks, and the second is the gate.

**The factored form is the pairwise form.** The implementation never builds an `N`-by-`N` matrix;
it sums squared batch means instead. That the two agree is an algebraic identity — the one Lean
records as `fourierRealizedAngularEnergy_featureForm` — so it should hold to machine precision.
A gap here means the weights, the einsum or the `1/D` are wrong, and nothing later would be
trustworthy.

**Unbiasedness.** Averaged over draws, the estimate must land on the kernel it approximates —
the exact angular kernel times the `K` radial cosines, which is what the feature path targets.
It is *not* the 3-image pairwise kernel; using that as the target would mix "is the sampler
unbiased" with "how do the cosines compare with the reflection", two different questions.

A wrong frequency scale, a wrong phase range or a missing `sqrt(2)` all break this check and
nothing else catches them.

In [ ]:
IDENTITY = cached("identity", lambda: TestFourierFactoredIdentity())
print()
UNBIASED = cached("unbiased", lambda: {f"{v}|{b}": r for (v, b), r in TestFourierUnbiased(
   n=int(CFG["check_n"]), d=TIME_D, n_features=int(CFG["n_features"]),
   reps=int(CFG["check_reps"])).items()})

_worst = max(abs(r["z"]) for r in UNBIASED.values())
print(f"\nlargest deviation across every batch and variant: {_worst:.2f} standard errors")
print("the estimator matches the identity and shows no bias" if _worst < 5. else
      "SOMETHING IS WRONG -- read the table above before going on")

## 3. Time — where the feature path passes the exact one

Only the repulsion term is timed, after the shared wristband map. The radial, angular and moment
terms are the same work for every path, so leaving them in would shift every column by the same
amount and hide what is being measured.

The prediction: the exact kernel costs about `N^2 d`, and a feature path costs about `N D (d + K)`.
Dividing one by the other, `d` mostly cancels and what is left is roughly `N / D`, so the feature
path wins once the batch passes the feature budget. The measured crossover normally sits well
above that estimate, because fixed per-call overhead dominates at small sizes. Report the measured
number, not the model.

Pairwise is skipped above the memory budget, with a line saying so.

In [ ]:
def _RunTiming():
   rows = TestFourierTiming(d=TIME_D, sizes=tuple(CFG["time_sizes"]),
                            n_features=int(CFG["n_features"]),
                            max_pairwise_gib=MAX_PAIRWISE_GIB)
   return {f"{name}|{n}": r for (name, n), r in rows.items()}


TIMING = cached("timing", _RunTiming)
_sizes = sorted({int(k.split("|")[1]) for k in TIMING})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, label in ((axes[0], "forward_ms", "forward"),
                       (axes[1], "backward_ms", "forward + backward")):
   for path in PATHS:
      xs = [n for n in _sizes if f"{path}|{n}" in TIMING]
      if not xs:
         continue
      ys = [TIMING[f"{path}|{n}"][key] for n in xs]
      ax.plot(xs, ys, "o-", color=COLOR[path], label=path,
              linewidth=2.6 if path.startswith("fourier") else 1.6)
   ax.set_xscale("log", base=2); ax.set_yscale("log")
   ax.set_xlabel("batch size N"); ax.set_ylabel("milliseconds")
   ax.set_title(f"({'ab'[label != 'forward']}) {label}")
   ax.set_xticks(_sizes); ax.set_xticklabels([str(s) for s in _sizes])
axes[0].legend()
fig.suptitle(f"Repulsion cost against batch size, d={TIME_D}, D={CFG['n_features']}", y=1.02)
fig.savefig(FIG_DIR / f"{PRESET}__timing.png")
plt.show()

print(f"{'N':>7} " + " ".join(f"{p:>16}" for p in PATHS))
for n in _sizes:
   cells = [f"{TIMING[f'{p}|{n}']['forward_ms']:.2f}" if f"{p}|{n}" in TIMING else "skipped"
            for p in PATHS]
   print(f"{n:>7} " + " ".join(f"{c:>16}" for c in cells))
for path in PATHS[1:]:
   both = [n for n in _sizes if f"{path}|{n}" in TIMING and f"pairwise|{n}" in TIMING]
   won = [n for n in both
          if TIMING[f"{path}|{n}"]["forward_ms"] < TIMING[f"pairwise|{n}"]["forward_ms"]]
   if won:
      up = (TIMING[f"pairwise|{max(both)}"]["forward_ms"]
            / TIMING[f"{path}|{max(both)}"]["forward_ms"])
      print(f"{path}: passes pairwise at N = {min(won)}; {up:.1f}x faster at N = {max(both)}")
   elif both:
      print(f"{path}: did not pass pairwise at any size both could run")
   else:
      print(f"{path}: pairwise never ran, so there is no comparison to make")

## 4. Features against accuracy

Four answers to "how many features", side by side. They are not in competition.

| tier | rule | status |
|---|---|---|
| proved, worst case | `D = 4 neumannSup^2 / (E0^2 delta eps^2)` | machine-checked |
| proved, batch-measured | `D = 4 radialEnergy^2 / (E0^2 delta eps^2)` | machine-checked |
| analytic | `D = rho / eps^2`, `rho_phase ~ (3/2) e^(2c^2/d) - 1` | analytic, not proved |
| measured | the `1 - delta` quantile of the error over repeated draws | measurement |

Everything is written in **relative** error, `eps = absolute error / E(mu_0)`, so the four sit on
one scale. The two proved rules hold for every distribution and use Chebyshev, so confidence
costs `1/delta` rather than `log(1/delta)` — twenty times on its own at 95%.

The gap between the proved rule and the measurement is large, and that is not a defect in either.
The proved rule is the one that never lies. The measured rule is the one to size a run with.
Quoting the measured number as if it were proved, or provisioning from the proved number, are
both mistakes.

In [ ]:
def _RunVariance():
   out = TestFourierVarianceAndD(d=TIME_D, features=tuple(CFG["var_features"]),
                                 reps=int(CFG["var_reps"]))
   out["rows"] = {f"{v}|{f}": r for (v, f), r in out["rows"].items()}
   return out


VARIANCE = cached("variance", _RunVariance)
_feats = sorted({int(k.split("|")[1]) for k in VARIANCE["rows"]})

fig, ax = plt.subplots(figsize=(6.5, 4.4))
for variant, colour in (("paired", COLOR["fourier paired"]), ("phase", COLOR["fourier phase"])):
   xs = [f for f in _feats if f"{variant}|{f}" in VARIANCE["rows"]]
   ys = [VARIANCE["rows"][f"{variant}|{f}"]["rel_sd"] for f in xs]
   ax.plot(xs, ys, "o-", color=colour, label=f"measured, {variant}")
   rho = VARIANCE[f"rho_{variant}"]
   ax.plot(xs, [math.sqrt(rho / f) for f in xs], "--", color=colour, alpha=0.6,
           label=f"analytic, {variant}")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("features D"); ax.set_ylabel("relative standard deviation of one estimate")
ax.set_title(f"Spread against the feature budget, d={TIME_D}")
ax.legend()
fig.savefig(FIG_DIR / f"{PRESET}__variance.png")
plt.show()

_top = max(_feats)
_rho_phase = VARIANCE["rows"][f"phase|{_top}"]["rho"]
_rho_paired = VARIANCE["rows"][f"paired|{_top}"]["rho"]
print(f"measured rho at D={_top}:  paired {_rho_paired:.5f}   phase {_rho_phase:.5f}   "
      f"({_rho_phase / max(_rho_paired, 1e-30):.1f}x apart)")
print(f"analytic:                 paired {VARIANCE['rho_paired']:.5f}   "
      f"phase {VARIANCE['rho_phase']:.5f}")
print(f"proved, batch-measured:   D = {VARIANCE['d_rad']:,.0f}")
print(f"proved, worst case:       D = {VARIANCE['d_sup']:,.0f}")
print(f"\nSizing rule to use: D = rho / eps^2 with rho measured at the working (c, d).")
for _eps in (0.20, 0.10, 0.085, 0.05):
   print(f"   eps {_eps:>5.3f}   paired D = {_rho_paired / _eps ** 2:>8.0f}   "
         f"phase D = {_rho_phase / _eps ** 2:>8.0f}")

## 5. What the harmonic cut cannot see

The `l <= 1` energy of any distribution on the sphere is fixed by two numbers: a constant, and the
length of the mean direction. So every arrangement whose mean direction is zero gets the same
value and the same gradient.

Five batches below. Four of them have a mean direction of zero and are plainly different from one
another: uniform on a great circle, a two-lobed belt, four clusters at the corners of a square, a
single antipodal pair. The exact kernel separates all four. The cut sees almost none of the
difference; what little it does see is a finite-batch term of order `1/sqrt(N)`, not a reading of
the arrangement.

This is the case the Fourier path exists to cover, and it is a bias, not noise: more steps do not
remove it.

In [ ]:
def _RunBlind():
   return TestFourierSeesWhatSpectralCannot(d=TIME_D, n_features=max(CFG["var_features"]))


BLIND = cached("blindness", _RunBlind)
_names = list(BLIND)
_short = [n.replace("uniform on ", "").replace("four clusters at the corners of a square",
                                               "four clusters") for n in _names]

fig, ax = plt.subplots(figsize=(8.5, 4.2))
_x = np.arange(len(_names)); _w = 0.27
ax.bar(_x - _w, [BLIND[n]["exact"] for n in _names], _w, label="exact", color="#8a8a8a")
ax.bar(_x, [BLIND[n]["spectral"] for n in _names], _w, label="ell <= 1 cut",
       color=COLOR["spectral"])
ax.bar(_x + _w, [BLIND[n]["fourier"] for n in _names], _w,
       yerr=[BLIND[n]["fourier_sd"] for n in _names], capsize=3,
       label="fourier features", color=COLOR["fourier paired"])
ax.set_xticks(_x); ax.set_xticklabels(_short, rotation=18, ha="right")
ax.set_ylabel("repulsion energy")
ax.set_title(f"Five arrangements of the directions, d={TIME_D}. Only the first is uniform.")
ax.legend()
fig.savefig(FIG_DIR / f"{PRESET}__blindness.png")
plt.show()

_blind = [n for n in _names if n != "uniform on the sphere"]
_spread = lambda key: max(BLIND[n][key] for n in _blind) - min(BLIND[n][key] for n in _blind)
print(f"spread across the four arrangements with a zero mean direction:")
print(f"   exact           {_spread('exact'):.6f}")
print(f"   ell <= 1 cut    {_spread('spectral'):.6f}  "
      f"({100. * _spread('spectral') / _spread('exact'):.1f}% of the real spread)")
print(f"   fourier         {_spread('fourier'):.6f}  "
      f"({100. * _spread('fourier') / _spread('exact'):.1f}%)")

## 6. Training — closeness to the target against wall-clock

Every path trains the same small MLP autoencoder from the same weights on the same batches, and
one instrument scores all of them on a held-out split.

**The model is deliberately small.** An earlier benchmark put every path behind a convolutional
encoder that took 83% of the step, so the largest speedup any loss could have shown was 1.2x, and
the numbers said more about the encoder than the loss. The share of the step spent in the loss is
measured and printed below; under about a third, this cell is comparing MLPs.

**On the latent width.** The model size and the latent width are separate knobs — a small MLP
reaches any `embed_dim` you ask for. What limits it is `in_dim`. A latent at least as wide as the
input lets the autoencoder pass the input through untouched, so the reconstruction term applies no
pressure and the run measures the wristband term against nothing; the test refuses that shape
rather than reporting it. Raise the two together, as the `full` preset does at
`in_dim=256, embed_dim=64`. It is worth doing: the gap between the paths grows with the latent
width, and the paired feature's margin over the random phase goes from roughly 3x at `d=16` to
roughly 20x at `d=128`.

Setting `MODEL = "full"` swaps the MLP for the attention-and-flow architecture that ships. That
answers a different question — whether the shipping latent geometry behaves the same — and it
makes the speed columns meaningless, which the loss share will show.

**Read the two panels together.** If (a) is flat and (b) is not, that is the result. If (a) is not
flat, the paths did not reach the same quality, and (b) is not yet a speed result — raise `D`.

Panel (c) is the honest one: the score reached by a given number of seconds. A cheaper path takes
more steps in the same time, and that is the whole claim.

In [ ]:
def _RunTraining():
   return TestFourierTrainingRun(
      batch_size=int(CFG["batch_size"]),
      n_features=int(CFG["n_features"]),
      n_samples=int(CFG["n_samples"]),
      n_epochs=int(CFG["n_epochs"]),
      seeds=tuple(CFG["seeds"]),
      calibration_reps=int(CFG["cal_reps"]),
      in_dim=int(CFG["in_dim"]),
      embed_dim=int(CFG["embed_dim"]),
      hidden=int(CFG["hidden"]),
      model=MODEL,
      paths=PATHS,                    # named, so pairwise is trained rather than assumed
      max_pairwise_gib=MAX_PAIRWISE_GIB,
   )


TRAIN = cached("training", _RunTraining)
MODE = "shared calibration"           # the fair one: every path on one loss scale
ROWS = TRAIN["results"][MODE]
RUN_PATHS = [p for p in PATHS if p in ROWS]


def _Spread(path, key):
   vals = [r["final"][key] for r in ROWS[path]["runs"]]
   return float(np.mean(vals)), (float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
_x = np.arange(len(RUN_PATHS))

means, errs = zip(*[_Spread(p, "angular") for p in RUN_PATHS])
axes[0].bar(_x, means, yerr=errs, capsize=4, color=[COLOR[p] for p in RUN_PATHS])
axes[0].axhline(TRAIN["floor"][0], ls="--", color="k", alpha=0.6,
                label="a true N(0, I) batch")
axes[0].set_xticks(_x); axes[0].set_xticklabels(RUN_PATHS, rotation=18, ha="right")
axes[0].set_ylabel("energy distance of the directions from uniform")
axes[0].set_title("(a) where each path landed"); axes[0].legend()

secs = [ROWS[p]["seconds"] for p in RUN_PATHS]
axes[1].bar(_x, secs, color=[COLOR[p] for p in RUN_PATHS])
axes[1].set_xticks(_x); axes[1].set_xticklabels(RUN_PATHS, rotation=18, ha="right")
axes[1].set_ylabel("seconds per epoch")
axes[1].set_title("(b) what that cost")

for path in RUN_PATHS:
   hist = ROWS[path]["runs"][0]["history"]
   axes[2].plot([h["seconds"] for h in hist], [h["angular"] for h in hist], "o-",
                color=COLOR[path], label=path,
                linewidth=2.6 if path.startswith("fourier") else 1.6)
axes[2].axhline(TRAIN["floor"][0], ls="--", color="k", alpha=0.6)
axes[2].set_xlabel("seconds of training"); axes[2].set_ylabel("energy distance from uniform")
axes[2].set_title("(c) the same score, against wall-clock"); axes[2].legend()

fig.suptitle(f"{MODE}, batch {CFG['batch_size']}, D {CFG['n_features']}, "
             f"{CFG['n_epochs']} epochs, {len(CFG['seeds'])} seeds", y=1.03)
fig.savefig(FIG_DIR / f"{PRESET}__training.png")
plt.show()

print(f"{'path':<16} {'loss share':>11} {'s/epoch':>9} {'speedup':>9} {'angle':>11} "
      f"{'radius KS':>11} {'dcor':>8} {'recon MSE':>11}")
for path in RUN_PATHS:
   share = TRAIN["shares"][path]["share"]
   up = ROWS["pairwise"]["seconds"] / ROWS[path]["seconds"] if "pairwise" in ROWS else float("nan")
   print(f"{path:<16} {100. * share:>10.1f}% {ROWS[path]['seconds']:>9.2f} {up:>8.2f}x "
         f"{ROWS[path]['angular']:>11.6f} {ROWS[path]['ks']:>11.5f} "
         f"{ROWS[path]['dcor']:>8.4f} {ROWS[path]['mse']:>11.5f}")
print(f"{'N(0, I) target':<16} {'':>11} {'':>9} {'':>9} {TRAIN['floor'][0]:>11.6f} "
      f"{TRAIN['floor'][2]:>11.5f} {TRAIN['floor'][4]:>8.4f}")

_share = min(TRAIN["shares"][p]["share"] for p in RUN_PATHS)
if _share < 0.33:
   print(f"\nWARNING  the cheapest loss is only {100. * _share:.1f}% of the step, so this table "
         f"is mostly about the MLP.\n         Raise batch_size, or shrink `hidden`, before "
         f"reading anything into the speed column.")

## 7. What this run does and does not show

Read the printed numbers, not this text — it is a guide to what to look for.

**What a good run looks like.** The identity holds to machine precision, no unbiasedness z-score
passes about 2, the Fourier path crosses the exact one somewhere between a few hundred and a few
thousand points, the paired variant is several times quieter than the random phase, and the
`l <= 1` cut sees a few percent of a spread the features track fully.

**Three things this does not settle.**

- **The kernel the guarantees are about.** The proved bounds are stated for the full Neumann image
  sum. The pairwise path in Python keeps three images. The pointwise gap between those two is
  proved; the lift of that bound to the energy level is still an open `sorry` in the Lean
  development. So "the pairwise path is exact" means exact for the 3-image kernel.
- **A frozen draw.** Every estimator theorem is about the mean over draws. Holding the frequencies
  fixed makes a kernel of rank `D` with a blind set of its own, which is the failure the whole
  path exists to avoid. `feature_seed` is for repeating a test, not for training.
- **The logarithm.** The energy estimate is unbiased; `(1/beta) log(estimate)` is not, because
  `log` is concave. The offset is about `-eps^2/2` and it errs toward tolerating a collapsed
  batch. Small at the usual settings, and a reason to prefer descending the energy itself where
  that choice exists.

Figures land in `figures/`, measurements in `results/<preset>__<name>.json`. Delete one result
file to recompute one measurement; set `FORCE_RECOMPUTE = True` to redo all of them.